# L1 - LLM Data Quality Assessment
Audit automatico della qualita dei dati Olist con supporto LLM.
Analizza 8 tabelle raw e genera report Markdown per ciascuna.

In [ ]:
import os, sys, json, requests
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown

_HERE          = Path(os.path.abspath('__file__')).parent
_ROOT          = _HERE.parent
RAW_PATH       = str(_ROOT / '1_raw_data') + os.sep
SCORECARD_PATH = str(_ROOT / '4_dq_scorecards') + os.sep
OUT_PATH       = str(_HERE / 'outputs') + os.sep
os.makedirs(OUT_PATH, exist_ok=True)
print('RAW_PATH      :', RAW_PATH)
print('SCORECARD_PATH:', SCORECARD_PATH)
print('OUT_PATH      :', OUT_PATH)

In [ ]:
LLM_PROVIDER      = 'ollama'
OLLAMA_MODEL      = 'llama3.2:3b'
OLLAMA_URL        = 'http://localhost:11434/api/chat'
OPENAI_MODEL      = 'gpt-4o-mini'
ANTHROPIC_MODEL   = 'claude-haiku-4-5-20251001'
OPENAI_API_KEY    = os.getenv('OPENAI_API_KEY', '')
ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY', '')

def call_llm(system_prompt, user_prompt):
    try:
        if LLM_PROVIDER == 'ollama':
            payload = {
                'model': OLLAMA_MODEL,
                'stream': False,
                'messages': [
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user',   'content': user_prompt}
                ]
            }
            r = requests.post(OLLAMA_URL, json=payload, timeout=120)
            r.raise_for_status()
            return r.json()['message']['content']
        elif LLM_PROVIDER == 'openai':
            import openai
            client = openai.OpenAI(api_key=OPENAI_API_KEY)
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {'role': 'system', 'content': system_prompt},
                    {'role': 'user',   'content': user_prompt}
                ])
            return resp.choices[0].message.content
        elif LLM_PROVIDER == 'anthropic':
            import anthropic
            client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
            msg = client.messages.create(
                model=ANTHROPIC_MODEL,
                max_tokens=1024,
                system=system_prompt,
                messages=[{'role': 'user', 'content': user_prompt}])
            return msg.content[0].text
    except Exception as e:
        return f'[LLM ERROR] {e}'

print(f'Provider attivo: {LLM_PROVIDER}')

In [ ]:
TABLES = {
    'orders':      ('olist_orders_dataset.csv',         'dq_scorecard_orders.csv'),
    'customers':   ('olist_customers_dataset.csv',      'dq_scorecard_customers.csv'),
    'order_items': ('olist_order_items_dataset.csv',    'dq_scorecard_order_items.csv'),
    'products':    ('olist_products_dataset.csv',       'dq_scorecard_products.csv'),
    'sellers':     ('olist_sellers_dataset.csv',        'dq_scorecard_sellers.csv'),
    'payments':    ('olist_order_payments_dataset.csv', 'dq_scorecard_order_payments.csv'),
    'reviews':     ('olist_order_reviews_dataset.csv',  'dq_scorecard_order_reviews.csv'),
    'geolocation': ('olist_geolocation_dataset.csv',    'dq_scorecard_geolocation.csv'),
}

datasets   = {}
scorecards = {}

for name, (raw_file, sc_file) in TABLES.items():
    raw_path = RAW_PATH + raw_file
    sc_path  = SCORECARD_PATH + sc_file
    if not os.path.exists(raw_path):
        print(f'  [SKIP] {raw_file} non trovato')
        continue
    df = pd.read_csv(raw_path)
    datasets[name] = df
    if os.path.exists(sc_path):
        scorecards[name] = pd.read_csv(sc_path)
    print(f'  [{name}] {len(df):,} righe x {df.shape[1]} colonne')

In [ ]:
SYSTEM_AUDIT = (
    'Sei un senior Data Quality Analyst specializzato in e-commerce. '
    'Stai analizzando i dati di Olist, marketplace brasiliano. '
    'Scrivi in italiano. Sii preciso, conciso e orientato al business. '
    'Usa sezioni Markdown: ## Panoramica, ## Problemi rilevati, ## Raccomandazioni.'
)

SYSTEM_RULES = (
    'Sei un esperto di data governance. Genera regole di qualita dati formali. '
    'Scrivi in italiano. Per ogni regola usa il formato: '
    'RULE_ID, Colonna, Tipo (Completezza|Unicita|Validita|Consistenza|Tempestivita), '
    'Descrizione, Soglia.'
)

SYSTEM_SUMMARY = (
    'Sei un data analyst che deve comunicare ai manager non tecnici. '
    'Scrivi in italiano, tono professionale ma accessibile. '
    'Produci un executive summary di max 150 parole per la tabella analizzata.'
)

print('Prompt caricati.')

In [ ]:
def build_profile(name, df):
    lines = []
    lines.append(f'Tabella: {name}')
    lines.append(f'Righe: {len(df):,} | Colonne: {df.shape[1]}')
    lines.append('')
    lines.append('Colonne e null:')
    for col in df.columns:
        null_n = df[col].isna().sum()
        null_p = null_n / len(df) * 100
        dtype  = str(df[col].dtype)
        lines.append(f'  - {col} [{dtype}]: {null_n} null ({null_p:.1f}%)')
    lines.append('')
    lines.append('Duplicati: ' + str(df.duplicated().sum()))
    num_cols = df.select_dtypes(include='number').columns
    if len(num_cols):
        lines.append('')
        lines.append('Statistiche numeriche (min/max/mean):')
        for col in num_cols[:6]:
            lines.append(f'  - {col}: min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}')
    return '\n'.join(lines)

def audit_table(name, df):
    profile  = build_profile(name, df)
    user_msg = f'Analizza la qualita dei dati di questa tabella Olist:\n\n{profile}'
    return call_llm(SYSTEM_AUDIT, user_msg)

def generate_rules(name, df):
    cols     = ', '.join(df.columns.tolist())
    nulls    = {c: f'{df[c].isna().mean()*100:.1f}%' for c in df.columns if df[c].isna().any()}
    user_msg = (f'Tabella: {name}\nColonne: {cols}\n'
                f'Colonne con null: {nulls}\n'
                f'Genera 5-8 regole di qualita dati formali per questa tabella.')
    return call_llm(SYSTEM_RULES, user_msg)

def stakeholder_summary(name, df, audit_text):
    user_msg = (f'Tabella: {name} ({len(df):,} righe)\n'
                f'Risultati audit tecnico:\n{audit_text[:800]}\n\n'
                f'Scrivi un executive summary per manager non tecnici.')
    return call_llm(SYSTEM_SUMMARY, user_msg)

print('Funzioni pronte.')

In [ ]:
results = {}

for name, df in datasets.items():
    print(f'Audit: {name}...')

    audit   = audit_table(name, df)
    rules   = generate_rules(name, df)
    summary = stakeholder_summary(name, df, audit)
    results[name] = {'audit': audit, 'rules': rules, 'summary': summary}

    # Mostra risultati inline nel notebook
    display(Markdown(f'---\n### {name.upper()}'))
    display(Markdown('**AUDIT TECNICO**\n\n' + audit))
    display(Markdown('**REGOLE DQ**\n\n' + rules))
    display(Markdown('**EXECUTIVE SUMMARY**\n\n' + summary))

    # Salva file Markdown
    out_file = OUT_PATH + f'dq_audit_{name}.md'
    with open(out_file, 'w', encoding='utf-8') as fout:
        fout.write(f'# DQ Audit - {name}\n\n')
        fout.write('## Audit Tecnico\n\n' + audit + '\n\n')
        fout.write('## Regole di Qualita\n\n' + rules + '\n\n')
        fout.write('## Executive Summary\n\n' + summary + '\n')

    print(f'  Salvato -> {out_file}')

print(f'Audit completato per {len(results)} tabelle.')

In [ ]:
# Report aggregato
agg_file = OUT_PATH + 'dq_audit_RIEPILOGO.md'
with open(agg_file, 'w', encoding='utf-8') as f:
    f.write('# Riepilogo DQ Audit - Olist Dataset\n\n')
    f.write(f'Tabelle analizzate: {len(results)}\n\n')
    f.write('| Tabella | Righe | Col. con null | Duplicati |\n')
    f.write('|---------|-------|---------------|-----------|\n')
    for name, df in datasets.items():
        null_cols = df.isna().any().sum()
        dupes     = df.duplicated().sum()
        f.write(f'| {name} | {len(df):,} | {null_cols} | {dupes} |\n')
    f.write('\n---\n\n')
    for name, res in results.items():
        f.write(f'## {name}\n\n')
        f.write('### Executive Summary\n\n')
        f.write(res['summary'] + '\n\n')
print(f'Riepilogo salvato: {agg_file}')